# 01 - Data Exploration

This notebook explores the data pipeline for RLHF fine-tuning:
- Loading and inspecting instruction-following datasets
- Data quality analysis
- Preference data structure
- Chat template formatting

In [ ]:
# Install dependencies if needed
# !pip install datasets transformers matplotlib seaborn pandas

In [ ]:
import sys
sys.path.append('..')

from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data.formatting import ChatTemplateFormatter, TemplateType
from data.curation import DataCurator, QualityConfig

## 1. Loading Instruction Datasets

We'll explore popular instruction-following datasets used for SFT.

In [ ]:
# Load Alpaca dataset
alpaca = load_dataset('tatsu-lab/alpaca', split='train')
print(f'Alpaca dataset size: {len(alpaca)}')
print(f'Columns: {alpaca.column_names}')

In [ ]:
# Examine sample
sample = alpaca[0]
print('Sample instruction:')
print(f"Instruction: {sample['instruction']}")
print(f"Input: {sample['input']}")
print(f"Output: {sample['output'][:200]}...")

## 2. Loading Preference Datasets

Preference datasets contain chosen/rejected pairs for reward modeling and DPO.

In [ ]:
# Load HH-RLHF preference dataset
hh_rlhf = load_dataset('Anthropic/hh-rlhf', split='train[:1000]')
print(f'HH-RLHF sample size: {len(hh_rlhf)}')
print(f'Columns: {hh_rlhf.column_names}')

In [ ]:
# Examine preference pair
sample = hh_rlhf[0]
print('Chosen response:')
print(sample['chosen'][:500])
print('\n' + '='*50 + '\n')
print('Rejected response:')
print(sample['rejected'][:500])

## 3. Data Quality Analysis

In [ ]:
# Analyze response lengths
chosen_lengths = [len(x['chosen']) for x in hh_rlhf]
rejected_lengths = [len(x['rejected']) for x in hh_rlhf]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(chosen_lengths, bins=50, alpha=0.7, label='Chosen')
axes[0].hist(rejected_lengths, bins=50, alpha=0.7, label='Rejected')
axes[0].set_xlabel('Response Length (chars)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Response Length Distribution')

axes[1].scatter(chosen_lengths, rejected_lengths, alpha=0.3)
axes[1].plot([0, max(chosen_lengths)], [0, max(chosen_lengths)], 'r--')
axes[1].set_xlabel('Chosen Length')
axes[1].set_ylabel('Rejected Length')
axes[1].set_title('Chosen vs Rejected Length')

plt.tight_layout()
plt.show()

## 4. Chat Template Formatting

In [ ]:
# Format instruction using different templates
instruction = "Write a haiku about machine learning."
response = "Neural networks learn\nPatterns hidden in the data\nWisdom emerges"

# This will use the ChatTemplateFormatter once implemented
print("Alpaca format:")
print(f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{response}""")

## 5. Next Steps

- Proceed to `02_sft_training.ipynb` for supervised fine-tuning
- Use the data curation module to filter and clean your data
- Generate synthetic data using the synthetic module if needed